# Paper10 compound-clustered BCa (Chem-style)

**Scratch** vs **FT Morgan learned fixmol** on **test RMSE**, over held-out molecules in each of the 11 paper datasets.

**Design (per dataset):**
1. Unit RMSE at `(molecule, context, log_dose, time)`; seed-average once.
2. Unit Δ = `RMSE(Scratch) − RMSE(FT)` (positive ⇒ FT better).
3. Unit win: `win_row = 1` if FT better on that unit.
4. Nested BCa over `context`, `time`, `log_dose` (`cluster=molecule`) for **Δ** and **win_rate** (same nesting).
5. **`win_rate_compound`:** average seed-avg scores within molecule over units, hard win if FT better; mean over molecules (no bootstrap).

**Scoring (parallel precompute):**
```bash
sbatch scripts/submit_paper10_unit_rmse_parallel.sbatch
```
Then keep `RUN_SCORING=False` and run Chem-style / bootstrap cells.


In [8]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

CHEM_SCRIPTS = Path("/home/icb/olga.novitskaia/Chem-PerturBridge_analysis/scripts")
sys.path.insert(0, str(CHEM_SCRIPTS))
from cluster_bootstrap_ci import (  # noqa: E402
    OBSERVED_COMPOUND_PANEL_SCOPE,
    cluster_bca_nested_mean_ci_table,
)

REPO = Path("/home/icb/olga.novitskaia/lpm_style")
RESULTS = REPO / "results"
OUTPUT_DIR = RESULTS / "paper10_bootstrap"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VENV_PYTHON = REPO / "lpm_training_venv" / "bin" / "python"
SCORE_SCRIPT = REPO / "scripts" / "paper10_compound_win_rates.py"

UNIT_LONG = OUTPUT_DIR / "paper10_unit_rmse_long_scratch_vs_ft_morgan_learned_fixmol_mol_ctx_dose_time.tsv"
SUMMARY_OUT = OUTPUT_DIR / "paper10_chem_style_summary_scratch_vs_ft_morgan_learned_fixmol_mol_ctx_dose_time.tsv"
UNIT_DELTA_OUT = OUTPUT_DIR / "paper10_chem_style_unit_deltas_scratch_vs_ft_morgan_learned_fixmol_mol_ctx_dose_time.tsv"
COMPOUND_OUT = OUTPUT_DIR / "paper10_chem_style_compound_wins_scratch_vs_ft_morgan_learned_fixmol_mol_ctx_dose_time.tsv"

REF_FAMILY = "scratch_target_only"
VAR_FAMILY = "finetune_morgan_learned_fixed_updated_embeddings"

TABLE11_ORDER = [
    "lincs_phase1",
    "lincs_phase2",
    "novartis",
    "vcpi_0001",
    "op3",
    "tahoe100",
    "cigs_tcm",
    "dilimap_train",
    "gdpx2",
    "sciplex",
    "cigs_mce",
]
DISPLAY = {
    "lincs_phase1": "LINCS Phase I",
    "lincs_phase2": "LINCS Phase II",
    "novartis": "Novartis DRUG-seq",
    "vcpi_0001": "VCPI vcpi-0001",
    "op3": "OP3",
    "tahoe100": "Tahoe-100M",
    "cigs_tcm": "CIGS TCM",
    "dilimap_train": "DILImap train",
    "gdpx2": "Ginkgo GDPx2",
    "sciplex": "sci-Plex",
    "cigs_mce": "CIGS MCE",
}

N_BOOT = 2000
BOOTSTRAP_SEED = 20260505
CI_LEVEL = 0.95

# Set True to (re)score unit RMSEs from Artur checkpoints.
RUN_SCORING = False  # use sbatch scripts/paper10_unit_rmse_dose_time.sbatch
MAX_SEEDS = None  # e.g. 1 for smoke test

try:
    display
except NameError:
    def display(x):
        print(x)


## 1. Load precomputed unit RMSE `(molecule, context, log_dose, time)`

Precompute with `sbatch scripts/submit_paper10_unit_rmse_parallel.sbatch` (GPU array + merge).


In [9]:
cmd_check = [
    str(VENV_PYTHON),
    str(SCORE_SCRIPT),
    "--check-only",
    "--output-dir",
    str(OUTPUT_DIR),
]
print("Running:", " ".join(cmd_check), flush=True)
proc = subprocess.run(cmd_check, cwd=str(REPO), capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)

access_path = OUTPUT_DIR / "paper10_checkpoint_access_check.tsv"
if access_path.is_file():
    access = pd.read_csv(access_path, sep="\t")
    display(access["status"].value_counts().to_frame("n"))


Running: /home/icb/olga.novitskaia/lpm_style/lpm_training_venv/bin/python /home/icb/olga.novitskaia/lpm_style/scripts/paper10_compound_win_rates.py --check-only --output-dir /home/icb/olga.novitskaia/lpm_style/results/paper10_bootstrap
Reading /ictstr01/home/icb/olga.novitskaia/lpm_style/results/lpm_paper10_results_current_check_long.tsv ...
Filtered long TSV rows: 220
status
ok    220
Saved /home/icb/olga.novitskaia/lpm_style/results/paper10_bootstrap/paper10_checkpoint_access_check.tsv



,n
status,
ok,220


In [10]:
RUN_SCORING = False
if RUN_SCORING:
    cmd = [
        str(VENV_PYTHON),
        str(SCORE_SCRIPT),
        "--output-dir",
        str(OUTPUT_DIR),
        "--datasets",
        *TABLE11_ORDER,
    ]
    if MAX_SEEDS is not None:
        cmd.extend(["--max-seeds", str(MAX_SEEDS)])
    print("Running unit scoring:", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=str(REPO))
    if proc.returncode != 0:
        raise RuntimeError(f"Scoring failed with exit code {proc.returncode}")
else:
    print(f"RUN_SCORING=False; expecting {UNIT_LONG}")

if not UNIT_LONG.is_file():
    raise FileNotFoundError(
        f"Missing {UNIT_LONG}\nSet RUN_SCORING=True after checkpoints are readable."
    )

scores = pd.read_csv(UNIT_LONG, sep="\t")
print(scores.head())
print(
    f"rows={len(scores)}  datasets={scores.dataset_slug.nunique()}  "
    f"molecules={scores.molecule.nunique()}  contexts={scores.context.nunique()}  "
    f"doses={scores.log_dose.nunique()}  times={scores.time.nunique()}  "
    f"seeds={sorted(scores.seed.unique())}"
)
need = {"log_dose", "time"}
missing = need - set(scores.columns)
if missing:
    raise KeyError(f"UNIT_LONG missing {sorted(missing)}; re-run dose/time scoring")

RUN_SCORING=False; expecting /home/icb/olga.novitskaia/lpm_style/results/paper10_bootstrap/paper10_unit_rmse_long_scratch_vs_ft_morgan_learned_fixmol_mol_ctx_dose_time.tsv
  dataset_slug   dataset                                      model_family  \
0     cigs_mce  CIGS MCE  finetune_morgan_learned_fixed_updated_embeddings   
1     cigs_mce  CIGS MCE  finetune_morgan_learned_fixed_updated_embeddings   
2     cigs_mce  CIGS MCE  finetune_morgan_learned_fixed_updated_embeddings   
3     cigs_mce  CIGS MCE  finetune_morgan_learned_fixed_updated_embeddings   
4     cigs_mce  CIGS MCE  finetune_morgan_learned_fixed_updated_embeddings   

   seed  molecule    context  log_dose  time      rmse  pooled_rmse  \
0    13  11982640  CVCL_0062       1.0  24.0  0.529374     0.572874   
1    13    157692  CVCL_0062       1.0  24.0  0.482338     0.572874   
2    13     20469  CVCL_0062       1.0  24.0  0.618604     0.572874   
3    13  20832634  CVCL_0062       1.0  24.0  0.548617     0.572874   
4   

## 2. Chem-style analysis

1. Seed-average unit RMSE once.
2. Precompute unit-level paired Δ and unit wins (`win_row`).
3. Nested BCa over `context`, `time`, `log_dose` for Δ and win rate (`cluster=molecule`).
4. `win_rate_compound`: hard molecule wins after averaging units (no bootstrap).


In [11]:
# Seed-average unit RMSE: one value per (dataset, family, molecule, context, log_dose, time)
UNIT_KEYS = ["dataset_slug", "dataset", "molecule", "context", "log_dose", "time"]
#INNER_COLS = ["context", "time", "log_dose"]
INNER_COLS = ["context", "time"]

unit_avg = (
    scores.groupby(
        ["dataset_slug", "dataset", "model_family", "molecule", "context", "log_dose", "time"],
        as_index=False,
    )["rmse"]
    .mean()
    .rename(columns={"rmse": "rmse_seedavg"})
)

# Wide: scratch vs FT on the same units
wide = (
    unit_avg.pivot_table(
        index=UNIT_KEYS,
        columns="model_family",
        values="rmse_seedavg",
        aggfunc="first",
    )
    .reset_index()
    .dropna(subset=[REF_FAMILY, VAR_FAMILY])
)
wide["delta_row"] = wide[REF_FAMILY] - wide[VAR_FAMILY]  # positive => FT better
wide["win_row"] = (wide[VAR_FAMILY] < wide[REF_FAMILY]).astype(float)

delta_frame = wide[
    UNIT_KEYS + ["delta_row", "win_row", REF_FAMILY, VAR_FAMILY]
].rename(columns={REF_FAMILY: "scratch_rmse_seedavg", VAR_FAMILY: "ft_rmse_seedavg"})

# Compound-level: average seed-avg scores over units, then hard win (no bootstrap).
comp = (
    wide.groupby(["dataset_slug", "dataset", "molecule"], as_index=False)
    .agg(
        scratch_rmse_seedavg=(REF_FAMILY, "mean"),
        ft_rmse_seedavg=(VAR_FAMILY, "mean"),
        n_units=("delta_row", "size"),
        n_contexts=("context", "nunique"),
        n_doses=("log_dose", "nunique"),
        n_times=("time", "nunique"),
    )
)
comp["delta_compound"] = comp["scratch_rmse_seedavg"] - comp["ft_rmse_seedavg"]
comp["win_compound"] = (comp["ft_rmse_seedavg"] < comp["scratch_rmse_seedavg"]).astype(float)

win_rate_compound = (
    comp.groupby(["dataset_slug", "dataset"], as_index=False)
    .agg(win_rate_compound=("win_compound", "mean"), n_compounds=("molecule", "nunique"))
)

print(delta_frame.shape, "unit-level Δ/win rows")
print(comp.shape, "compound rows")
display(delta_frame.head(3))
display(comp.head(3))
display(win_rate_compound.head(3))


(20610, 10) unit-level Δ/win rows
(1012, 11) compound rows


model_family,dataset_slug,dataset,molecule,context,log_dose,time,delta_row,win_row,scratch_rmse_seedavg,ft_rmse_seedavg
0,cigs_mce,CIGS MCE,72,CVCL_0062,1.0,24.0,-0.002187,0.0,0.497787,0.499974
1,cigs_mce,CIGS MCE,72,CVCL_0063,1.0,24.0,-0.006052,0.0,0.480061,0.486113
2,cigs_mce,CIGS MCE,125,CVCL_0062,1.0,24.0,0.008014,1.0,0.432044,0.424030


,dataset_slug,dataset,molecule,scratch_rmse_seedavg,ft_rmse_seedavg,n_units,n_contexts,n_doses,n_times,delta_compound,win_compound
0,cigs_mce,CIGS MCE,72,0.488924,0.493043,2,2,1,1,-0.004119,0.0
1,cigs_mce,CIGS MCE,125,0.454999,0.454385,2,2,1,1,0.000615,1.0
2,cigs_mce,CIGS MCE,525,0.484943,0.476947,2,2,1,1,0.007996,1.0


,dataset_slug,dataset,win_rate_compound,n_compounds
0,cigs_mce,CIGS MCE,0.317647,340
1,cigs_tcm,CIGS TCM,0.535354,99
2,dilimap_train,DILImap train,0.590909,22


In [12]:
# Nested BCa on unit Δ: cluster=molecule, inner=context×time×dose (per dataset).
delta_ci = cluster_bca_nested_mean_ci_table(
    delta_frame,
    group_cols=["dataset_slug", "dataset"],
    metric_cols={"delta_row": "delta_row"},
    cluster_col="molecule",
    inner_cols=INNER_COLS,
    outer_cols=None,
    n_boot=N_BOOT,
    ci_level=CI_LEVEL,
    seed=BOOTSTRAP_SEED,
    summary_level="paper10_dataset_metric",
    uncertainty_scope=OBSERVED_COMPOUND_PANEL_SCOPE,
)

# Win-rate BCa: same nesting as Δ (unit wins).
win_ci = cluster_bca_nested_mean_ci_table(
    delta_frame,
    group_cols=["dataset_slug", "dataset"],
    metric_cols={"win_rate": "win_row"},
    cluster_col="molecule",
    inner_cols=INNER_COLS,
    outer_cols=None,
    n_boot=N_BOOT,
    ci_level=CI_LEVEL,
    seed=BOOTSTRAP_SEED,
    summary_level="paper10_dataset_winrate_nested_unit",
    uncertainty_scope=OBSERVED_COMPOUND_PANEL_SCOPE,
)

delta_ci = delta_ci.rename(
    columns={
        "mean": "delta_hat",
        "ci_low": "delta_ci_low",
        "ci_high": "delta_ci_high",
        "ci_half_width": "delta_ci_half_width",
        "ci_method": "delta_ci_method",
        "ci_status": "delta_ci_status",
        "n_bootstrap_valid": "delta_n_bootstrap_valid",
        "n_compounds": "n_compounds",
        "metric": "bca_metric_label",
    }
)
win_ci = win_ci.rename(
    columns={
        "mean": "win_rate",
        "ci_low": "win_rate_ci_low",
        "ci_high": "win_rate_ci_high",
        "ci_half_width": "win_rate_ci_half_width",
        "ci_method": "win_rate_ci_method",
        "ci_status": "win_rate_ci_status",
        "n_bootstrap_valid": "win_rate_n_bootstrap_valid",
        "metric": "bca_metric_label",
    }
)

merge_keys = ["dataset_slug", "dataset"]
summary = delta_ci[
    merge_keys
    + [
        "delta_hat",
        "delta_ci_low",
        "delta_ci_high",
        "delta_ci_half_width",
        "delta_ci_method",
        "delta_ci_status",
        "delta_n_bootstrap_valid",
        "n_compounds",
        "n_rows",
        "n_finite_rows",
        "inner_strata",
        "cluster_col",
        "ci_level",
        "n_bootstrap_iterations",
        "uncertainty_scope",
    ]
].merge(
    win_ci[
        merge_keys
        + [
            "win_rate",
            "win_rate_ci_low",
            "win_rate_ci_high",
            "win_rate_ci_half_width",
            "win_rate_ci_method",
            "win_rate_ci_status",
            "win_rate_n_bootstrap_valid",
        ]
    ],
    on=merge_keys,
    how="inner",
).merge(
    win_rate_compound[merge_keys + ["win_rate_compound"]],
    on=merge_keys,
    how="left",
)

summary["excludes_zero"] = summary.apply(
    lambda r: bool(r["delta_ci_low"] > 0 or r["delta_ci_high"] < 0)
    if np.isfinite(r["delta_ci_low"])
    else False,
    axis=1,
)
summary["win_rate_excludes_0.5"] = summary.apply(
    lambda r: bool(r["win_rate_ci_low"] > 0.5 or r["win_rate_ci_high"] < 0.5)
    if np.isfinite(r["win_rate_ci_low"])
    else False,
    axis=1,
)
summary["ref_method"] = "Scratch"
summary["var_method"] = "FT Morgan learned fixmol"
summary["metric"] = "delta_mean_test_rmse_chem_nested"
summary["design"] = "chem_style_unit_delta_and_win_nested_ctx_time_dose_seedavg_first"

# Stable table-11 order
order_map = {slug: i for i, slug in enumerate(TABLE11_ORDER)}
summary["_ord"] = summary["dataset_slug"].map(order_map)
summary = summary.sort_values("_ord").drop(columns=["_ord"]).reset_index(drop=True)

summary.to_csv(SUMMARY_OUT, sep="\t", index=False)
delta_frame.to_csv(UNIT_DELTA_OUT, sep="\t", index=False)
comp.to_csv(COMPOUND_OUT, sep="\t", index=False)

show_cols = [
    "dataset",
    "n_compounds",
    "delta_hat",
    "delta_ci_low",
    "delta_ci_high",
    "excludes_zero",
    "win_rate",
    "win_rate_ci_low",
    "win_rate_ci_high",
    "win_rate_excludes_0.5",
    "win_rate_compound",
    "delta_ci_method",
    "inner_strata",
]
display(
    summary[show_cols].style.format(
        {
            "delta_hat": "{:+.6f}",
            "delta_ci_low": "{:.6f}",
            "delta_ci_high": "{:.6f}",
            "win_rate": "{:.3f}",
            "win_rate_ci_low": "{:.3f}",
            "win_rate_ci_high": "{:.3f}",
            "win_rate_compound": "{:.3f}",
        }
    )
)
print(f"\nSaved {SUMMARY_OUT}")
print(f"Saved {UNIT_DELTA_OUT}")
print(f"Saved {COMPOUND_OUT}")
print("delta = nested mean of unit (Scratch−FT) RMSE; positive => FT better")
print("win_rate = nested mean of unit wins + BCa (same nesting as delta_hat)")
print("win_rate_compound = no-bootstrap fraction of molecules with unit-averaged RMSE(FT) < RMSE(Scratch)")
print("units = molecule × context × log_dose × time")


,dataset,n_compounds,delta_hat,delta_ci_low,delta_ci_high,excludes_zero,win_rate,win_rate_ci_low,win_rate_ci_high,win_rate_excludes_0.5,win_rate_compound,delta_ci_method,inner_strata
0,LINCS Phase I,194,+0.009017,0.006639,0.011241,True,0.623,0.582,0.656,True,0.655,bca,"context,time"
1,LINCS Phase II,142,+0.008364,0.004581,0.013900,True,0.568,0.512,0.623,True,0.732,bca,"context,time"
2,Novartis DRUG-seq,145,+0.002795,0.002087,0.003799,True,0.821,0.781,0.852,True,0.862,bca,"context,time"
3,VCPI vcpi-0001,2,+0.000089,0.000040,0.000137,True,0.583,0.500,0.667,False,1.000,percentile_fallback,"context,time"
4,OP3,10,+0.004067,0.000728,0.008832,True,0.725,0.475,0.875,False,0.800,bca,"context,time"
5,Tahoe-100M,34,+0.002474,-0.005323,0.006486,False,0.598,0.520,0.664,True,0.676,bca,"context,time"
6,CIGS TCM,99,+0.000162,-0.000211,0.000537,False,0.474,0.415,0.531,False,0.535,bca,"context,time"
7,DILImap train,22,+0.000117,-0.000615,0.000695,False,0.517,0.368,0.640,False,0.591,bca,"context,time"
8,Ginkgo GDPx2,7,-0.001636,-0.024448,0.006424,False,0.679,0.399,0.827,False,0.714,bca,"context,time"
9,sci-Plex,17,-0.001010,-0.008800,0.002141,False,0.469,0.240,0.529,False,0.294,bca,"context,time"



Saved /home/icb/olga.novitskaia/lpm_style/results/paper10_bootstrap/paper10_chem_style_summary_scratch_vs_ft_morgan_learned_fixmol_mol_ctx_dose_time.tsv
Saved /home/icb/olga.novitskaia/lpm_style/results/paper10_bootstrap/paper10_chem_style_unit_deltas_scratch_vs_ft_morgan_learned_fixmol_mol_ctx_dose_time.tsv
Saved /home/icb/olga.novitskaia/lpm_style/results/paper10_bootstrap/paper10_chem_style_compound_wins_scratch_vs_ft_morgan_learned_fixmol_mol_ctx_dose_time.tsv
delta = nested mean of unit (Scratch−FT) RMSE; positive => FT better
win_rate = nested mean of unit wins + BCa (same nesting as delta_hat)
win_rate_compound = no-bootstrap fraction of molecules with unit-averaged RMSE(FT) < RMSE(Scratch)
units = molecule × context × log_dose × time


In [13]:
print(1)

1
